In [ ]:
!pip install -U langchain langchain-google-genai langchain-tavily tavily-python

In [ ]:
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")
SEARCH_API_KEY = userdata.get("TAVILY_API_KEY")


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=API_KEY
)

response = model.invoke("What is your knowledge cutoff date")
print(response.content)

My knowledge cutoff is **early 2023**.

This means I don't have information about events or developments that have occurred since that time. My responses are based on the vast amount of text and code I was trained on, which only goes up to that point.


In [ ]:
!pip install langchain langchain_community

In [ ]:
from typing import List
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.tools import tool
from tavily import TavilyClient
from langgraph.graph import MessageGraph, END

from langgraph.graph import MessageGraph, END

tavily = TavilyClient(api_key=SEARCH_API_KEY)

# ----- Tools -----

@tool
def parse_resume(text: str) -> str:
    """
    Extracts skills, roles, years of experience from resume text
    """
    prompt = f"Extract structured info like skills, roles, years of experience from this resume:\n\n{text}"
    return model.invoke([HumanMessage(content=prompt)]).content


@tool
def search_jobs_tavily(query: str) -> str:
    """
    Searches for jobs using Tavily and returns summarized results.
    """
    results = tavily.search(query)
    print(results)
    return results["results"]

# ---- Nodes ----

def profile_node(state: List[BaseMessage]) -> List[BaseMessage]:
    resume = state[-1].content
    profile = parse_resume.invoke(resume)

    return state + [
        AIMessage(content=f"Extracted Profile:\n{profile}")
    ]


def job_search_node(state: List[BaseMessage]) -> List[BaseMessage]:
    profile = state[-1].content

    query = model.invoke([
        HumanMessage(
            content=f"""
Based on the following candidate profile, generate a concise job search query.

Profile:
{profile}

The query should be short and optimized for job search engines.
Return only the search query.
"""
        )
    ])

    jobs = search_jobs_tavily.invoke(query.content)

    return state + [
        AIMessage(content=f"Found Jobs:\n{jobs}")
    ]


def matcher_node(state: List[BaseMessage]) -> List[BaseMessage]:
    profile = state[-2].content
    jobs = state[-1].content

    prompt = f"""
Given the candidate profile and the list of job results below:

Candidate Profile:
{profile}

Job Listings:
{jobs}

Rank and recommend the top 3 most suitable jobs for this candidate.
Explain briefly why each job matches the profile.
"""

    ranked = model.invoke([HumanMessage(content=prompt)])

    return state + [ranked]
def cover_letter_node(state: List[BaseMessage]) -> List[BaseMessage]:
    profile = state[-3].content
    best_job = state[-1].content

    prompt = f"""Write a personalized short cover letter for this job:

Job Info:
{best_job}

Based on Profile:
{profile}
"""

    letter = model.invoke([HumanMessage(content=prompt)])

    return state + [letter]


# ---- LangGraph ----
# ---- LangGraph ----

graph = MessageGraph()

graph.add_node("profile", profile_node)
graph.add_node("search", job_search_node)
graph.add_node("match", matcher_node)
graph.add_node("cover", cover_letter_node)

graph.set_entry_point("profile")

graph.add_edge("profile", "search")
graph.add_edge("search", "match")
graph.add_edge("match", "cover")
graph.add_edge("cover", END)

input_msg = HumanMessage(
    content="I'm a frontend developer with 3 years of experience in React, TypeScript, and HTML."
)

app=graph.compile()
result = app.invoke([input_msg])

print("\n--- Output ---\n")

for msg in result:
    print(f"{type(msg).__name__}: {msg.content}\n")

/tmp/ipykernel_1583/2344400644.py:107: LangGraphDeprecatedSinceV10: MessageGraph is deprecated in LangGraph v1.0.0, to be removed in v2.0.0. Please use StateGraph with a `messages` key instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph = MessageGraph()


{'query': 'Frontend Developer React TypeScript', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://jobs.lever.co/relay/298f0f7a-d493-49fa-935d-89a76ef09665', 'title': 'Relay - Frontend Developer (React, TypeScript) - Lever', 'content': 'We are looking for talented and motivated front-end engineers of all experience levels. Relay engineers thrive in a high-growth, high-velocity environment where success depends on the highest levels of communication, code quality, and pragmatism. Each engineer must be able to individually own a slice of the Relay codebase while also maintaining a working knowledge of others. Accordingly, Relay is looking for team members who are not only highly-skilled individual contributors but also strong collaborators and communicators. + Rapidly implement functional UI elements from design mocks and workflow diagrams, with an eye toward correctness and readability. + 3+ years of experience building production React applications